# attention mechanisms

this notebook explores attention mechanisms including coding self-attention, causal attention, and multi-head attention

In [1]:
# create an input embedding

import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # your
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55],  # step
])

print("input shape:", inputs.shape)
print(inputs)

input shape: torch.Size([6, 3])
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])


In [3]:
# select "journey" as the query

query = inputs[1]

print("query:", query)
print("query shape:", query.shape)

query: tensor([0.5500, 0.8700, 0.6600])
query shape: torch.Size([3])


In [ ]:
# calculate attention scores (unnormalized comparisons)

attention_scores = torch.empty(inputs.shape[0])

for index, input_vector in enumerate(inputs):
    attention_scores[index] = torch.dot(input_vector, query)

print("attention scores:", attention_scores)

attention scores: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [13]:
# normalize attention scores to produce attention weights

attention_weights = torch.softmax(attention_scores, dim=0)

print("attention weights:", attention_weights)
print("weighted sum:", attention_weights.sum())

attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
weighted sum: tensor(1.)


In [19]:
# calculate conctext vector

context_vector = torch.zeros(query.shape)

for index, input_vector in enumerate(inputs):
    context_vector += (
        attention_weights[index]*input_vector
    )

print("context_vector:",context_vector)

context_vector: tensor([0.4419, 0.6515, 0.5683])


In [21]:
# calculate all pairwise scores using matrix multiplication

attention_scores = inputs @ inputs.T

print("score shape:", attention_scores.shape)
print(attention_scores)

score shape: torch.Size([6, 6])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [25]:
# normalize every row in attention scores

attention_weights = torch.softmax(
    attention_scores,
    dim=-1,
)

print("attention weight shape:", attention_weights.shape)
print("row sums:", attention_weights.sum(dim=-1))

attention weight shape: torch.Size([6, 6])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [27]:
# calculate every context vector

context_vectors = attention_weights @ inputs

print("context shape:", context_vectors.shape)
print(context_vectors)

context shape: torch.Size([6, 3])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [28]:
# define a self-attention model

import torch.nn as nn

class self_attention(nn.Module):
    def __init__(self, input_dimension, output_dimension, qkv_bias=False):
        super().__init__()

        self.query_layer = nn.Linear(
            input_dimension,
            output_dimension,
            bias=qkv_bias,
        )

        self.key_layer = nn.Linear(
            input_dimension,
            output_dimension,
            bias=qkv_bias,
        )

        self.value_layer = nn.Linear(
            input_dimension,
            output_dimension,
            bias=qkv_bias,
        )

    def forward(self, inputs):
        queries = self.query_layer(inputs)
        keys = self.key_layer(inputs)
        values = self.value_layer(inputs)

        attention_scores = queries @ keys.T

        scale = keys.shape[-1] ** 0.5

        attention_weights = torch.softmax(
            attention_scores / scale,
            dim=-1,
        )

        context_vectors = (
            attention_weights @ values
        )

        return context_vectors

In [29]:
# create and run the module

torch.manual_seed(789)

attention_layer = self_attention(
    input_dimension=3,
    output_dimension=2,
)

context_vectors = attention_layer(inputs)

print("context vectors:", context_vectors)

context vectors: tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [31]:
# inspect trainable parameters

for name, parameter in attention_layer.named_parameters():
    print(name)
    print("shape:", parameter.shape)
    print("requires gradient:", parameter.requires_grad)
    print()

query_layer.weight
shape: torch.Size([2, 3])
requires gradient: True

key_layer.weight
shape: torch.Size([2, 3])
requires gradient: True

value_layer.weight
shape: torch.Size([2, 3])
requires gradient: True



In [35]:
# creating the casual-attention class

import torch
import torch.nn as nn

class casaul_attention(nn.Module):
    def __init__(
            self,
            d_in,
            d_out,
            context_length,
            dropout,
            qkv_bias=False
    ):
        super().__init__()

        self.w_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        attention_scores = (
            queries
            @ keys.transpose(1, 2)
        )

        mask = self.mask.bool()[
            :num_tokens,
            :num_tokens
        ]

        attention_scores.masked_fill_(
            mask,
            -torch.inf,
        )

        attention_weights = torch.softmax(
            attention_scores
            / keys.shape[-1] ** 0.5,
            dim=-1,
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context_vectors = (
            attention_weights
            @ values
        )

        return context_vectors

In [36]:
# create a multi-head attention class

class multi_head_attention(nn.Module):
    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False,
    ):
        super().__init__()

        assert d_out % num_heads == 0

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.w_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.w_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias,
        )

        self.out_proj = nn.Linear(
            d_out,
            d_out,
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(
                    context_length,
                    context_length,
                ),
                diagonal=1,
            ),
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.w_query(x)
        keys = self.w_key(x)
        values = self.w_value(x)

        queries = queries.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        keys = keys.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        values = values.view(
            batch_size,
            num_tokens,
            self.num_heads,
            self.head_dim,
        )

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = (
            queries
            @ keys.transpose(2, 3)
        )

        mask = self.mask.bool()[
            :num_tokens,
            :num_tokens
        ]

        attention_scores.masked_fill_(
            mask,
            -torch.inf,
        )

        attention_weights = torch.softmax(
            attention_scores
            / keys.shape[-1] ** 0.5,
            dim=-1,
        )

        attention_weights = self.dropout(
            attention_weights
        )

        context_vectors = (
            attention_weights
            @ values
        )

        context_vectors = (
            context_vectors.transpose(1, 2)
        )

        context_vectors = (
            context_vectors
            .contiguous()
            .view(
                batch_size,
                num_tokens,
                self.d_out,
            )
        )

        context_vectors = self.out_proj(
            context_vectors
        )

        return context_vectors

In [37]:
# running an example with multi-head attention

torch.manual_seed(789)

multi_head_attention_layer = multi_head_attention(
    d_in=3,
    d_out=4,          
    context_length=6,
    dropout=0.0,
    num_heads=2,
)

batch_inputs = inputs.unsqueeze(0)

context_vectors = multi_head_attention_layer(batch_inputs)

print("input shape:", batch_inputs.shape)
print("context vectors shape:", context_vectors.shape)
print(context_vectors)

input shape: torch.Size([1, 6, 3])
context vectors shape: torch.Size([1, 6, 4])
tensor([[[-0.4802,  0.0977, -0.5124, -0.4274],
         [-0.5080,  0.0521, -0.4516, -0.3720],
         [-0.5178,  0.0368, -0.4312, -0.3530],
         [-0.5034,  0.0426, -0.4134, -0.3306],
         [-0.5076,  0.0431, -0.4081, -0.3126],
         [-0.4979,  0.0452, -0.4007, -0.3102]]], grad_fn=<ViewBackward0>)
